# Invsqrt candidate — full official verification (CPU-only)

Build the experimental piecewise-linear seed circuit, run the exhaustive classical fixed-point test, compare circuit resources with the Newton–Raphson reference, and run the official verifier over the complete benchmark domain. GPU is not requested.

**Full verifier workload:** 3,133,452 inputs × 64 shots = 200,540,928 simulated shots. This can take a long time on CPU. Logs, metrics, the circuit, and a ZIP bundle are written to `/kaggle/working/invsqrt-full-results` so GitHub Actions can retrieve them.

In [ ]:
from pathlib import Path
import datetime
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import traceback
import zipfile

WORK = Path('/kaggle/working')
RESULTS = WORK / 'invsqrt-full-results'
RESULTS.mkdir(parents=True, exist_ok=True)
REPO = WORK / 'quantum-source'
REPO_URL = 'https://github.com/evaristebernhard/quantum.git'
BRANCH = 'kaggle/invsqrt-full-run'
N_P = 10
SHOTS_PER_X = 64
BATCH_SIZE = 256
MAX_X = 3 * (2**N_P - 2)**2
EXPECTED_SHOTS = MAX_X * SHOTS_PER_X
WORKERS = max(1, min(4, os.cpu_count() or 1))
started = datetime.datetime.now(datetime.timezone.utc).isoformat()
manifest = {
    'task': 'google-verifiable-quantum-circuit-compilation-invsqrt',
    'mode': 'CPU-only',
    'repository': REPO_URL,
    'branch': BRANCH,
    'n_p': N_P,
    'max_x': MAX_X,
    'shots_per_x': SHOTS_PER_X,
    'expected_verified_shots': EXPECTED_SHOTS,
    'batch_size': BATCH_SIZE,
    'max_workers': WORKERS,
    'python': sys.version,
    'platform': platform.platform(),
    'cpu_count': os.cpu_count(),
    'started_at_utc': started,
}
summary = {'status': 'NOT_STARTED'}
failure = None
print(json.dumps(manifest, indent=2), flush=True)


In [ ]:
try:
    if REPO.exists():
        shutil.rmtree(REPO)
    clone = subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)],
        text=True, capture_output=True
    )
    (RESULTS / 'git-clone.log').write_text((clone.stdout or '') + '\n' + (clone.stderr or ''), encoding='utf-8')
    if clone.returncode:
        raise RuntimeError('Could not clone the candidate branch: ' + (clone.stderr or '')[-3000:])
    source_sha = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
    manifest['source_commit'] = source_sha
    verifier_dir = REPO / 'google-verifiable-quantum-circuit-compilation' / 'verifiers' / 'invsqrt'
    kickmix_dir = REPO / 'google-verifiable-quantum-circuit-compilation' / 'kickmix'
    env = os.environ.copy()
    env['PYTHONPATH'] = str(kickmix_dir) + os.pathsep + str(verifier_dir)
    env['PYTHONUNBUFFERED'] = '1'

    # Install only the small Python dependencies needed by the test harness.
    deps = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pytest'], text=True, capture_output=True)
    (RESULTS / 'dependency-install.log').write_text((deps.stdout or '') + '\n' + (deps.stderr or ''), encoding='utf-8')
    if deps.returncode:
        raise RuntimeError('Dependency installation failed: ' + (deps.stderr or '')[-3000:])

    probe = subprocess.run(
        [sys.executable, '-c', 'import kickmix; print("KickMix import OK")'],
        cwd=verifier_dir, env=env, text=True, capture_output=True
    )
    (RESULTS / 'kickmix-import.log').write_text((probe.stdout or '') + '\n' + (probe.stderr or ''), encoding='utf-8')
    if probe.returncode:
        raise RuntimeError('KickMix native module could not be imported: ' + (probe.stderr or '')[-3000:])

    # Exhaustive classical model check over every benchmark input.
    test = subprocess.run(
        [sys.executable, '-m', 'pytest', '-q', '-s', 'test_piecewise_linear.py'],
        cwd=verifier_dir, env=env, text=True, capture_output=True
    )
    test_log = (test.stdout or '') + '\n' + (test.stderr or '')
    (RESULTS / 'classical-model-test.log').write_text(test_log, encoding='utf-8')
    print('CLASSICAL MODEL TEST\n' + test_log[-5000:], flush=True)
    if test.returncode:
        raise RuntimeError('Exhaustive classical fixed-point model test failed.')

    # Save a side-by-side resource report; this proxy is not the official score.
    comparison = subprocess.run(
        [sys.executable, 'compare_invsqrt_resources.py', '--n-p', str(N_P)],
        cwd=verifier_dir, env=env, text=True, capture_output=True
    )
    (RESULTS / 'resource-comparison.json').write_text(comparison.stdout or '', encoding='utf-8')
    (RESULTS / 'resource-comparison.stderr.txt').write_text(comparison.stderr or '', encoding='utf-8')
    if comparison.returncode:
        raise RuntimeError('Circuit resource comparison failed: ' + (comparison.stderr or '')[-3000:])
    print('RESOURCE COMPARISON\n' + (comparison.stdout or '')[-5000:], flush=True)

    # Build the n_p=10 candidate circuit; the full classical check was already run above.
    circuit_path = RESULTS / 'candidate-invsqrt.kmb'
    build = subprocess.run(
        [sys.executable, 'piecewise_linear_circuit.py', str(circuit_path), '--n-p', str(N_P), '--skip-model', '--format', 'kmb'],
        cwd=verifier_dir, env=env, text=True, capture_output=True
    )
    build_log = (build.stdout or '') + '\n' + (build.stderr or '')
    (RESULTS / 'circuit-build.log').write_text(build_log, encoding='utf-8')
    print('CIRCUIT BUILD\n' + build_log[-3000:], flush=True)
    if build.returncode or not circuit_path.is_file():
        raise RuntimeError('Candidate circuit build failed.')

    # Full official verification: all x=1..MAX_X, with 64 shots per x.
    command = [
        sys.executable, 'verify_submission_invsqrt.py', str(circuit_path),
        '--format', 'kmb', '--n-p', str(N_P), '--shots-per-x', str(SHOTS_PER_X),
        '--batch-size', str(BATCH_SIZE), '--max-workers', str(WORKERS)
    ]
    manifest['verifier_command'] = command
    manifest['verifier_started_at_utc'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    verifier_log_path = RESULTS / 'official-verifier.log'
    print('Starting full official verifier. Expected shots:', EXPECTED_SHOTS, flush=True)
    verify_started = time.monotonic()
    with verifier_log_path.open('w', encoding='utf-8') as log_file:
        process = subprocess.Popen(command, cwd=verifier_dir, env=env, stdout=log_file, stderr=subprocess.STDOUT, text=True)
        last_notice = 0
        while process.poll() is None:
            time.sleep(60)
            elapsed = int(time.monotonic() - verify_started)
            if elapsed - last_notice >= 300:
                last_notice = elapsed
                print(f'Official verifier still running: {elapsed // 60} minutes elapsed; log is being saved.', flush=True)
        return_code = process.wait()
    verifier_text = verifier_log_path.read_text(encoding='utf-8', errors='replace')
    manifest['verifier_exit_code'] = return_code
    manifest['verifier_duration_seconds'] = round(time.monotonic() - verify_started, 2)
    manifest['verifier_finished_at_utc'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    print('OFFICIAL VERIFIER LOG TAIL\n' + verifier_text[-8000:], flush=True)
    lines = verifier_text.splitlines()
    json_start = next((i for i in range(len(lines) - 1, -1, -1) if lines[i].strip() == '{'), None)
    if json_start is not None:
        try:
            summary = json.loads('\n'.join(lines[json_start:]))
        except json.JSONDecodeError:
            summary = {'status': 'NO_REPORT', 'note': 'Could not parse verifier JSON; inspect official-verifier.log.'}
    else:
        summary = {'status': 'NO_REPORT', 'note': 'Verifier produced no JSON summary; inspect official-verifier.log.'}
    summary['evaluator_exit_code'] = return_code
    summary['verified_shots_expected'] = EXPECTED_SHOTS
    summary['source_commit'] = source_sha
    if return_code != 0 or summary.get('status') != 'PASS':
        raise RuntimeError('Full official verifier did not PASS; see summary.json and official-verifier.log.')
except Exception as exc:
    failure = f'{type(exc).__name__}: {exc}'
    if summary.get('status') != 'PASS':
        summary['status'] = 'FAIL'
    summary['failure'] = failure
    summary['traceback'] = traceback.format_exc()
finally:
    manifest['finished_at_utc'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    manifest['final_status'] = 'PASS' if failure is None else 'FAIL'
    (RESULTS / 'run-manifest.json').write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
    (RESULTS / 'summary.json').write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')
    bundle_path = RESULTS / 'invsqrt-full-verification-results.zip'
    with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
        for path in sorted(RESULTS.rglob('*')):
            if path.is_file() and path != bundle_path:
                bundle.write(path, path.relative_to(RESULTS))
    print('FINAL SUMMARY\n' + json.dumps(summary, indent=2), flush=True)
    print('Result files:', [p.name for p in RESULTS.iterdir() if p.is_file()], flush=True)

if failure is not None:
    raise RuntimeError(failure)
print('PASS: full official invsqrt verification completed.')
